In [ ]:
import numpy as np
import random

num_points = int(input("Enter total number of waste collection points (excluding depot): "))
VEHICLE_CAPACITY = float(input("Enter maximum vehicle capacity (weight/volume): "))

coordinates = {}
print("\nEnter coordinates for Depot (Point 0):")
depot_x = float(input("  Depot X: "))
depot_y = float(input("  Depot Y: "))
coordinates[0] = (depot_x, depot_y)

demands = {0: 0}

for i in range(1, num_points + 1):
    print(f"\nEnter data for Collection Point {i}:")
    x = float(input(f"  Point {i} X: "))
    y = float(input(f"  Point {i} Y: "))
    demand = float(input(f"  Point {i} Waste Demand: "))

    coordinates[i] = (x, y)
    demands[i] = demand

num_nodes = len(coordinates)
distances = np.zeros((num_nodes, num_nodes))

for i in range(num_nodes):
    for j in range(num_nodes):
        if i != j:
            distances[i][j] = np.linalg.norm(np.array(coordinates[i]) - np.array(coordinates[j]))
        else:
            distances[i][j] = float('inf')

NUM_ANTS = 10
ALPHA = 1.0
BETA = 2.0
RHO = 0.1
TAU_0 = 1.0
MAX_ITERATIONS = 50

pheromones = np.full((num_nodes, num_nodes), TAU_0)

global_best_distance = float('inf')
global_best_fleet_size = float('inf')
global_best_routes = []

for iteration in range(MAX_ITERATIONS):
    all_ant_routes = []
    all_ant_distances = []
    all_ant_fleet_sizes = []

    for ant in range(NUM_ANTS):
        unvisited = set(range(1, num_nodes))
        ant_fleet_routes = []

        while unvisited:
            current_route = [0]
            current_load = 0
            current_node = 0

            while True:
                feasible = [node for node in unvisited if current_load + demands[node] <= VEHICLE_CAPACITY]

                if not feasible:
                    break

                probabilities = []
                for target in feasible:
                    tau = pheromones[current_node][target] ** ALPHA
                    eta = (1.0 / distances[current_node][target]) ** BETA
                    probabilities.append(tau * eta)

                sum_prob = sum(probabilities)
                probabilities = [p / sum_prob for p in probabilities]

                next_node = random.choices(feasible, weights=probabilities, k=1)[0]

                current_route.append(next_node)
                current_load += demands[next_node]
                unvisited.remove(next_node)
                current_node = next_node

            current_route.append(0)
            ant_fleet_routes.append(current_route)

        all_ant_routes.append(ant_fleet_routes)

        total_dist = 0
        for route in ant_fleet_routes:
            for idx in range(len(route) - 1):
                total_dist += distances[route[idx]][route[idx+1]]

        all_ant_distances.append(total_dist)
        all_ant_fleet_sizes.append(len(ant_fleet_routes))

        if (len(ant_fleet_routes) < global_best_fleet_size) or \
           (len(ant_fleet_routes) == global_best_fleet_size and total_dist < global_best_distance):
            global_best_distance = total_dist
            global_best_fleet_size = len(ant_fleet_routes)
            global_best_routes = ant_fleet_routes

    pheromones *= (1.0 - RHO)

    for ant in range(NUM_ANTS):
        quality_score = 100.0 / (all_ant_distances[ant] * all_ant_fleet_sizes[ant])
        for route in all_ant_routes[ant]:
            for idx in range(len(route) - 1):
                i, j = route[idx], route[idx+1]
                pheromones[i][j] += quality_score

print("\n" + "=" * 50)
print(f"OPTIMIZATION COMPLETE ({MAX_ITERATIONS} Iterations performed)")
print("=" * 50)
print(f"Minimum Number of Vehicles Required: {global_best_fleet_size}")
print(f"Minimum Total Travel Distance: {global_best_distance:.2f} units")
print("\nBest Vehicle Route Paths:")
for truck_idx, route in enumerate(global_best_routes, 1):
    route_str = " -> ".join(["Depot" if node == 0 else f"Point_{node}" for node in route])
    print(f"  Truck {truck_idx}: {route_str}")
print("=" * 50)


Enter total number of waste collection points (excluding depot): 3
Enter maximum vehicle capacity (weight/volume): 5

Enter coordinates for Depot (Point 0):
  Depot X: 0
  Depot Y: 0

Enter data for Collection Point 1:
  Point 1 X: 0
  Point 1 Y: 4
  Point 1 Waste Demand: 3

Enter data for Collection Point 2:
  Point 2 X: 3
  Point 2 Y: 0
  Point 2 Waste Demand: 3

Enter data for Collection Point 3:
  Point 3 X: 0
  Point 3 Y: -4
  Point 3 Waste Demand: 2

OPTIMIZATION COMPLETE (50 Iterations performed)
Minimum Number of Vehicles Required: 2
Minimum Total Travel Distance: 20.00 units

Best Vehicle Route Paths:
  Truck 1: Depot -> Point_3 -> Point_2 -> Depot
  Truck 2: Depot -> Point_1 -> Depot
